# OpenPlaque — LCX vs OM Explicit AV-Groove Locus Identity
Fresh standalone Colab from the frozen baseline. Builds an anatomy-derived LA/LV groove locus and compares C6 vs C7 downstream of the established Family-2 split.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR = DRIVE_ROOT + '/LCX_AV_Groove_Locus_Identity_v1'
BRANCH = 'lcx-av-groove-locus-identity-from-main'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
print('Branch:', BRANCH)
print('Output:', OUTPUT_DIR)


In [ ]:
import os, shutil, subprocess, sys
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',repo],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',repo,'pytest','SimpleITK','scipy','pandas','matplotlib'],check=True)
head=subprocess.check_output(['git','-C',repo,'rev-parse','HEAD'],text=True).strip()
print('HEAD:',head)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-c',"import openplaque; from openplaque.lcx_av_groove_locus_identity import synthetic_groove_locus_self_test; print(openplaque.__file__); print(synthetic_groove_locus_self_test())"],check=True)
subprocess.run([sys.executable,'-m','pytest','-q','/content/OpenPlaque/tests/test_lcx_av_groove_locus_identity.py'],check=True)


In [ ]:
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_atrium_left.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_ventricle_left.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_atrium_right.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_ventricle_right.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_myocardium.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'Joint_Three_Vessel_Template_Classifier_v1/LCX_joint_candidate_ranking.csv',
 root/'LCX_Parent_Continuation_Topology_v1/summary.json',
]+[root/f'Joint_Three_Vessel_Template_Classifier_v1/candidate_{i:02d}_source_path.csv' for i in range(1,6)]
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))


In [ ]:
import os, subprocess, sys
runner=f'''\nfrom openplaque.lcx_av_groove_locus_identity import run\nr=run(r"{DRIVE_ROOT}", r"{OUTPUT_DIR}")\nprint("STATUS:",r["summary"]["status"])\nprint("CONTROLS:",r["summary"].get("groove_locus_controls"))\nprint("DECISION:",r["summary"].get("decision"))\nprint("C6:",r["summary"].get("C6_metrics"))\nprint("C7:",r["summary"].get("C7_metrics"))\nprint("REPORT:",r.get("report"))\nprint("ZIP:",r.get("zip"))\n'''
env=os.environ.copy(); env.update({'OPENBLAS_NUM_THREADS':'1','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','NUMEXPR_NUM_THREADS':'1'})
subprocess.run([sys.executable,'-c',runner],check=True,env=env)
